<a href="https://colab.research.google.com/github/do-alisa/final-data-analysis-project/blob/main/Spotify_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Loading the Spotify Dataset

In [2]:
# Install KaggleHub
!pip -q install kagglehub

import kagglehub
import pandas as pd
import numpy as np
import os
import glob

# Download dataset from Kaggle
dataset_path = kagglehub.dataset_download(
    "yashdev01/spotify-tracks-dataset"
)

print("Dataset location:", dataset_path)

# Find CSV files
csv_files = glob.glob(dataset_path + "/**/*.csv", recursive=True)
print(csv_files)

# Load the first CSV file
df = pd.read_csv(csv_files[0])

print("Dataset successfully loaded!")

Using Colab cache for faster access to the 'spotify-tracks-dataset' dataset.
Dataset location: /kaggle/input/spotify-tracks-dataset
['/kaggle/input/spotify-tracks-dataset/spotify-tracks-dataset.csv']
Dataset successfully loaded!


### Milestone 0 - Choose a Dataset

**DATASET**: Spotify Tracks Dataset

**SOURCE**: https://www.kaggle.com/datasets/yashdev01/spotify-tracks-dataset

**DESCRIPTION**:
A collection of Spotify tracks containing information about artists, albums,
song characteristics, popularity, and genre classifications. The dataset
can be used to analyze which musical features are associated with track
popularity.

**DATA DICTIONARY**:

- track_id: Unique identifier for each track
- artists: Artist or artists who performed the track
- album_name: Name of the album containing the track
- track_name: Name of the song
- popularity: Spotify popularity score from 0 to 100
- duration_ms: Track duration in milliseconds
- explicit: Indicates whether the track contains explicit content
- danceability: How suitable the track is for dancing, from 0 to 1
- energy: Intensity and activity level of the track, from 0 to 1
- key: Musical key of the track, represented by an integer from -1 to 11
- loudness: Overall loudness of the track measured in decibels
- mode: Musical modality; 1 represents major and 0 represents minor
- speechiness: Amount of spoken words in the track, from 0 to 1
- acousticness: Confidence that the track is acoustic, from 0 to 1
- instrumentalness: Likelihood that the track contains no vocals, from 0 to 1
- liveness: Likelihood that the track was recorded during a live performance, from 0 to 1
- valence: Musical positivity or happiness of the track, from 0 to 1
- tempo: Estimated tempo of the track in beats per minute
- time_signature: Estimated number of beats in each measure
- track_genre: Genre associated with the track

## Initial Quality Audit

In [13]:
# Remove unnecessary index columns before the audit
df = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], errors="ignore")

print("INITIAL QUALITY AUDIT")
print("---------------------")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\nNote: Originally, there were 22 columns. The columns Unnamed: 0.1 and Unnamed: 0 were excluded from the data dictionary \nbecause they were unnecessary index columns created during file export.")


print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nMissing values:")
missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("No missing values")
else:
    for column, count in missing.items():
        percentage = count / len(df) * 100
        print(f"- {column}: {count:,} missing ({percentage:.2f}%)")

print("\nDuplicates:")
print(f"- Exact duplicate rows: {df.duplicated().sum():,}")

# Number of repeated duplicate rows
repeated_track_rows = df["track_id"].duplicated().sum()

# Number of unique track IDs that appear more than once
repeated_track_ids = (
    df["track_id"].value_counts().gt(1).sum()
)

print(f"- Repeated track rows: {repeated_track_rows:,}")
print(f"- Unique track IDs appearing multiple times: {repeated_track_ids:,}")
print(f"- Unique track IDs: {df['track_id'].nunique():,}")


print("\nClarification on Repeated Track IDs:")
print("The dataset contains 24,259 repeated track IDs. These were not automatically removed because \nthe repeated rows may contain different genre labels for the same song. \nThe rows were retained for genre analysis, while one row per track was used for track-level statistical analysis.")

print("\nPotential problems:")
print("- Some tracks appear under multiple genre labels.")
print("- Exact duplicate rows should be removed.")
print("- Numeric columns should be checked for unusual or invalid values.")

# Show example repeated tracks
print("\nExamples of tracks with multiple genre labels:")

display(
    df[df["track_id"].duplicated(keep=False)]
    [["track_id", "track_name", "artists", "track_genre"]]
    .sort_values("track_id")
    .head(10)
)

INITIAL QUALITY AUDIT
---------------------
Rows: 114,000
Columns: 20

Note: Originally, there were 22 columns. The columns Unnamed: 0.1 and Unnamed: 0 were excluded from the data dictionary 
because they were unnecessary index columns created during file export.

Column names:
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

Data types:
float64    9
object     5
int64      5
bool       1
Name: count, dtype: int64

Missing values:
- artists: 1 missing (0.00%)
- album_name: 1 missing (0.00%)
- track_name: 1 missing (0.00%)

Duplicates:
- Exact duplicate rows: 450
- Repeated track rows: 24,259
- Unique track IDs appearing multiple times: 16,641
- Unique track IDs: 89,741

Clarification on Repeated Track IDs:
The dataset contains 24,259 repeated track IDs. These were not automat

,track_id,track_name,artists,track_genre
15028,001APMDOl3qtx1526T11n1,Better,Pink Sweat$;Kirby,chill
103211,001APMDOl3qtx1526T11n1,Better,Pink Sweat$;Kirby,soul
85578,001YQlnDSduXd5LgBd66gT,El Tiempo Es Dinero - Remasterizado 2007,Soda Stereo,punk-rock
100420,001YQlnDSduXd5LgBd66gT,El Tiempo Es Dinero - Remasterizado 2007,Soda Stereo,ska
91801,003vvx7Niy0yvhvHt4a68B,Mr. Brightside,The Killers,rock
3257,003vvx7Niy0yvhvHt4a68B,Mr. Brightside,The Killers,alternative
2106,003vvx7Niy0yvhvHt4a68B,Mr. Brightside,The Killers,alt-rock
33178,004h8smbIoAkUNDJvVKwkG,Lovemark,Ouse;Powfu,emo
94239,004h8smbIoAkUNDJvVKwkG,Lovemark,Ouse;Powfu,sad
97533,006rHBBNLJMpQs8fRC2GDe,Agora Estou Sofrendo - Ao Vivo,Calcinha Preta;Gusttavo Lima,sertanejo
